# 08 - RS-PPO gegen ArmoRM: bewusst zirkulaerer Upper-Bound-Test

Dieses Notebook trainiert PPO bewusst gegen ArmoRM und evaluiert danach wieder gegen ArmoRM. Das ist zirkulaer und kein Proxy-Validitaets-Experiment.

## Was dieses Notebook zeigen kann und was nicht

**Ungueltig durch Zirkularitaet - nicht berichten:**

- Keine RQ2-Proxy-Validitaet: keine Aussage der Form "R ist ein valider Proxy fuer Reward".
- Keine Spearman-Gate-Zahl als Proxy-Validierung. Falls Rho berechnet wird, ist es reine Diagnostik und wird als `circular_do_not_report: true` markiert.
- Keine Generalisierung ueber ArmoRM hinaus.

**Gueltig trotz Zirkularitaet:**

- Obere Schranke: PPO gegen r_i macht delta_j ungefaehr zum Reward-Gradienten-Schritt. Wenn f(p,R) in diesem Best-Case lambda=p nicht schlaegt, ist der Negativbefund staerker.
- R-minus: Bricht PPO die konfliktfreie Geometrie?
- Wall A / R2: Bleibt U_p(theta(lambda)) auf quality-Achsen nichtlinear?
- LMC: Sind theta_SFT und PPO-Spezialisten linear verbunden?

Ein bindender Lauf wird vorregistriert. Ergebnis in beide Richtungen akzeptieren; keine Retrain-schauen-Retrain-Schleife.

## Setup

Repository vorbereiten, Dependencies installieren, Seeds setzen und Config drucken.

In [ ]:
%cd /content

import os, sys, json, math, random, shutil, subprocess, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

repo_path = Path('/content/master-thesis')
repo_url = 'https://github.com/NZhang137/master-thesis.git'
if (repo_path / '.git').is_dir():
    print('Repository exists; pulling latest changes.')
    subprocess.run(['git', '-C', str(repo_path), 'pull', '--ff-only'], check=False)
else:
    print('Repository missing; cloning from GitHub.')
    if repo_path.exists():
        shutil.rmtree(repo_path)
    subprocess.run(['git', 'clone', repo_url, str(repo_path)], check=True)

%cd /content/master-thesis

!pip install -q "transformers==4.40.0" "peft==0.10.0" "accelerate==0.29.3" "trl==0.8.6" bitsandbytes datasets scipy numpy pandas matplotlib safetensors

import numpy as np
import pandas as pd
import torch

CONFIG = {
    'SEED': 137,
    'OUTPUT_DIR': 'results/rs_ppo_armorm_circular',
    'OUTPUT_ZIP': 'rs_ppo_armorm_circular_outputs.zip',
    'BASE_MODEL': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    'ARMORM_MODEL': 'RLHFlow/ArmoRM-Llama3-8B-v0.1',
    'ATTRIBUTES': ['helpfulness', 'correctness', 'coherence', 'complexity', 'verbosity'],
    'CIRCULAR_ARMORM_ACKNOWLEDGED': True,
    'RUN_HEAD_SANITY': True,
    'RUN_SFT': False,
    'RUN_PPO': False,
    'RUN_GEOMETRY': False,
    'RUN_LMC': False,
    'RUN_REWARD_COLLECTION': False,
    'RUN_FINAL_MERGE': False,
    'HEAD_SANITY_SAMPLES': 200,
    'HEAD_SANITY_SPLIT': 'validation',
    'HEAD_STD_MIN': 1e-6,
    'HEAD_UNIQUE_MIN': 10,
    'REWARD_NUM_PROMPTS': 80,
    'REWARD_PROMPT_SPLIT': 'validation',
    'REWARD_PROMPT_OFFSET': 160,
    'MAX_NEW_TOKENS': 96,
    'REPETITION_PENALTY': 1.15,
    'NO_REPEAT_NGRAM_SIZE': 5,
    'PPO_AXES': ['helpfulness', 'correctness', 'coherence', 'complexity', 'verbosity'],
    'PPO_BATCH_SIZE': 64,
    'PPO_TOTAL_STEPS': 200,
    'PPO_N_PROMPTS': 2005,
    'M1PLUS_RHO': 0.5,
    'BOOTSTRAP_N': 2000,
    'BOOTSTRAP_SEED': 137,
    'SEARCH_SET_SEED': 137,
    'SEARCH_SET_DIRICHLET': 64,
    'LMC_GRID': [0.0, 0.25, 0.5, 0.75, 1.0],
    'MIN_GPU_MEMORY_GB': 35.0,
}

PROJECT_ROOT = Path.cwd().resolve()
OUTPUT_DIR = (PROJECT_ROOT / CONFIG['OUTPUT_DIR']).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(CONFIG['SEED']); np.random.seed(CONFIG['SEED']); torch.manual_seed(CONFIG['SEED'])
if torch.cuda.is_available(): torch.cuda.manual_seed_all(CONFIG['SEED'])

assert CONFIG['CIRCULAR_ARMORM_ACKNOWLEDGED'] is True
assert CONFIG['SEED'] == 137
print('Project root:', PROJECT_ROOT)
print('Output dir:', OUTPUT_DIR)
print(json.dumps(CONFIG, indent=2, sort_keys=True))

## Import, Firewall-Test und Pre-Registration

`train_rs_ppo.py` wird importiert. Die Firewall muss ohne Flag hart fehlschlagen und mit Flag eine laute Zirkularitaetswarnung liefern. Danach wird `preregistration.json` geschrieben.

In [ ]:
import importlib
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
rs_ppo = importlib.import_module('scripts.train_rs_ppo')
rs_ppo = importlib.reload(rs_ppo)

rs_ppo.CFG['out_dir'] = str(OUTPUT_DIR / 'rs_runs')
rs_ppo.CFG['batch_size'] = CONFIG['PPO_BATCH_SIZE']
rs_ppo.CFG['total_ppo_steps'] = CONFIG['PPO_TOTAL_STEPS']
rs_ppo.CFG['n_prompts'] = CONFIG['PPO_N_PROMPTS']
rs_ppo.CFG['armorm_model'] = CONFIG['ARMORM_MODEL']
rs_ppo.CFG['armorm_load_in_4bit'] = True
for axis in CONFIG['ATTRIBUTES']:
    rs_ppo.HELD_OUT_RM[axis] = CONFIG['ARMORM_MODEL']

try:
    rs_ppo.check_reward_firewall('helpfulness', CONFIG['ARMORM_MODEL'], circular_armorm_acknowledged=False)
    raise AssertionError('Firewall did not block unacknowledged ArmoRM PPO.')
except AssertionError as error:
    print('Firewall correctly blocks unacknowledged ArmoRM PPO:', error)

firewall_ack = rs_ppo.check_reward_firewall('helpfulness', CONFIG['ARMORM_MODEL'], circular_armorm_acknowledged=True)
assert firewall_ack['circularity_acknowledged'] is True
assert 'RQ2 (proxy validity)' in firewall_ack['retired_research_questions']
print('Acknowledged circularity warning:', firewall_ack['warning'])

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    assert path.exists(), f'Missing JSON output: {path}'

def write_numpy(path, values):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.save(path, np.asarray(values))
    assert path.exists(), f'Missing NumPy output: {path}'

PREREGISTRATION_PATH = OUTPUT_DIR / 'preregistration.json'
pre_registration = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'experiment': 'RS-faithful PPO against ArmoRM, deliberately circular upper-bound test',
    'seed': CONFIG['SEED'],
    'circularity': {
        'acknowledged': True,
        'retired_research_questions': ['RQ2 (proxy validity)'],
        'rationale': 'PPO reward model and evaluation model are both ArmoRM. This retires proxy-validity claims but defines an upper-bound test for f(p,R).',
    },
    'primary': {
        'metric': 'Delta U_p(lambda*_M1+ vs lambda=p)',
        'success': 'mean Delta U_p > 0 with bootstrap 95% CI excluding 0',
        'bootstrap_n': CONFIG['BOOTSTRAP_N'],
        'bootstrap_seed': CONFIG['BOOTSTRAP_SEED'],
        'binding_run': True,
    },
    'secondary_non_circular': {'R_minus_nonzero': 'yes/no', 'wall_A_R2': 'per axis', 'LMC': 'smooth yes/no'},
    'negative_control': 'quality-heavy preferences must run',
    'interpretation_rule': {
        'quality_success': 'Wall A was not a hard cap; strong positive result',
        'quality_failure': 'upper-bound negative result; endpoint-linear rules fail on nonlinear axes',
    },
    'valid_claims': ['upper bound', 'R-minus', 'wall A (R2)', 'LMC'],
    'invalid_claims': ['proxy validity', 'generalization beyond this reward model'],
    'config': CONFIG,
}
write_json(PREREGISTRATION_PATH, pre_registration)
print('Wrote:', PREREGISTRATION_PATH)

## Phase 0 - ArmoRM Head-Sanity und VRAM-Check

ArmoRM wird in 4-bit geladen. Jeder der fuenf HelpSteer2-Heads muss auf mindestens 200 Antworten variieren und mehr als 10 unterschiedliche Werte liefern. Bei Fehlschlag wird nicht trainiert.

In [ ]:
HEAD_SANITY_PATH = OUTPUT_DIR / 'head_sanity.json'
ARMORM_HELPSTEER_OBJECTIVES = {
    'helpfulness': (0, 'helpsteer-helpfulness'),
    'correctness': (1, 'helpsteer-correctness'),
    'coherence': (2, 'helpsteer-coherence'),
    'complexity': (3, 'helpsteer-complexity'),
    'verbosity': (4, 'helpsteer-verbosity'),
}
objective_indices = [ARMORM_HELPSTEER_OBJECTIVES[a][0] for a in CONFIG['ATTRIBUTES']]
objective_names = [ARMORM_HELPSTEER_OBJECTIVES[a][1] for a in CONFIG['ATTRIBUTES']]
print('ArmoRM HelpSteer mapping:', dict(zip(CONFIG['ATTRIBUTES'], objective_names)))

if CONFIG['RUN_HEAD_SANITY']:
    from datasets import load_dataset
    from scipy.stats import pearsonr
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig
    assert torch.cuda.is_available(), 'ArmoRM PPO path needs CUDA.'
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    total_gb = total_bytes / 1024**3
    print(f'GPU: {torch.cuda.get_device_name(0)} total={total_gb:.1f}GB')
    assert total_gb >= CONFIG['MIN_GPU_MEMORY_GB'], f'A100-40GB class runtime expected; got {total_gb:.1f}GB.'

    reward_tokenizer = AutoTokenizer.from_pretrained(CONFIG['ARMORM_MODEL'], use_fast=True)
    if reward_tokenizer.pad_token_id is None:
        reward_tokenizer.pad_token = reward_tokenizer.eos_token
    quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True)
    reward_model = AutoModelForSequenceClassification.from_pretrained(CONFIG['ARMORM_MODEL'], trust_remote_code=True, device_map='auto', torch_dtype=torch.bfloat16, quantization_config=quantization_config)
    reward_model.requires_grad_(False); reward_model.eval()

    ds = load_dataset('nvidia/HelpSteer2', split=CONFIG['HEAD_SANITY_SPLIT'])
    rng = np.random.default_rng(CONFIG['SEED'])
    idx = rng.choice(len(ds), size=CONFIG['HEAD_SANITY_SAMPLES'], replace=False)
    labels, chat_texts = [], []
    for i in idx:
        row = ds[int(i)]
        labels.append([float(row[a]) for a in CONFIG['ATTRIBUTES']])
        messages = [{'role': 'user', 'content': str(row['prompt'])}, {'role': 'assistant', 'content': str(row['response'])}]
        chat_texts.append(reward_tokenizer.apply_chat_template(messages, tokenize=False) if getattr(reward_tokenizer, 'chat_template', None) else f"Human: {row['prompt']}\n\nAssistant: {row['response']}")
    labels = np.asarray(labels, dtype=np.float64)

    scores = []
    device = next(reward_model.parameters()).device
    for start in range(0, len(chat_texts), 8):
        stop = min(start + 8, len(chat_texts))
        inputs = reward_tokenizer(chat_texts[start:stop], return_tensors='pt', padding=True, truncation=True, max_length=4096)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.inference_mode():
            outputs = reward_model(**inputs)
        rewards = getattr(outputs, 'rewards', None)
        if rewards is None:
            raise RuntimeError('ArmoRM output has no .rewards tensor.')
        tensor = torch.as_tensor(rewards).detach().float()
        if tensor.ndim == 1:
            tensor = tensor.unsqueeze(0)
        scores.append(tensor[:, objective_indices].cpu().numpy())
        if stop % 50 == 0 or stop == len(chat_texts):
            print(f'Scored {stop}/{len(chat_texts)} head-sanity samples')
    scores = np.concatenate(scores, axis=0).astype(np.float64)
    assert scores.shape == (CONFIG['HEAD_SANITY_SAMPLES'], len(CONFIG['ATTRIBUTES']))
    assert np.all(np.isfinite(scores))

    stds = scores.std(axis=0)
    unique_counts = [int(len(np.unique(scores[:, i]))) for i in range(scores.shape[1])]
    for i, axis in enumerate(CONFIG['ATTRIBUTES']):
        assert stds[i] > CONFIG['HEAD_STD_MIN'], f'ArmoRM head {axis} is constant: std={stds[i]}'
        assert unique_counts[i] > CONFIG['HEAD_UNIQUE_MIN'], f'ArmoRM head {axis} has too few unique values: {unique_counts[i]}'

    cross = np.full((5, 5), np.nan, dtype=np.float64)
    for i in range(5):
        for j in range(5):
            if np.std(scores[:, i]) > 0 and np.std(labels[:, j]) > 0:
                cross[i, j] = float(pearsonr(scores[:, i], labels[:, j]).statistic)
    head_sanity = {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'n': int(scores.shape[0]), 'attributes': CONFIG['ATTRIBUTES'], 'objective_names': objective_names, 'std': dict(zip(CONFIG['ATTRIBUTES'], stds.tolist())), 'unique_counts': dict(zip(CONFIG['ATTRIBUTES'], unique_counts)), 'armorm_vs_helpsteer2_pearson': cross.tolist(), 'diagonal': dict(zip(CONFIG['ATTRIBUTES'], np.diag(cross).tolist())), 'passed': True, 'circularity_acknowledged': True}
    write_json(HEAD_SANITY_PATH, head_sanity)
    write_numpy(OUTPUT_DIR / 'head_sanity_scores.npy', scores)
    write_numpy(OUTPUT_DIR / 'head_sanity_labels.npy', labels)
    display(pd.DataFrame(cross, index=CONFIG['ATTRIBUTES'], columns=CONFIG['ATTRIBUTES']))
else:
    write_json(HEAD_SANITY_PATH, {'passed': False, 'skipped': True, 'reason': 'RUN_HEAD_SANITY is False'})
    print('Head sanity skipped.')

## Phase 1 und 2 - SFT-Basis und PPO-Laeufe

SFT und PPO werden ueber `scripts/train_rs_ppo.py` gestartet. PPO nutzt ArmoRM nur mit `--circular_armorm_acknowledged`.

In [ ]:
if CONFIG['RUN_SFT']:
    subprocess.run([sys.executable, 'scripts/train_rs_ppo.py', '--phase', 'sft'], check=True)
else:
    print('RUN_SFT is False; SFT phase not started.')

ppo_commands = []
for axis in CONFIG['PPO_AXES']:
    ppo_commands.append([sys.executable, 'scripts/train_rs_ppo.py', '--phase', 'ppo', '--axis', axis, '--reward_model', CONFIG['ARMORM_MODEL'], '--circular_armorm_acknowledged'])

if CONFIG['RUN_PPO']:
    head_sanity = json.loads(HEAD_SANITY_PATH.read_text(encoding='utf-8'))
    assert head_sanity.get('passed') is True, 'Head sanity did not pass; do not train.'
    for cmd in ppo_commands:
        print('Running:', ' '.join(cmd))
        subprocess.run(cmd, check=True)
else:
    print('RUN_PPO is False; planned commands:')
    for cmd in ppo_commands:
        print('  ' + ' '.join(cmd))

## Phase 3 - Geometrie: Delta-Normen, R, R-minus und Floor-LP

Effektive LoRA-Updates werden geometrisch verglichen. Dieser Teil ist nicht zirkulaer, weil er keine Reward-Queries nutzt.

In [ ]:
DELTA_NORMS_PATH = OUTPUT_DIR / 'delta_norms.json'
GEOMETRY_PRECHECK_PATH = OUTPUT_DIR / 'geometry_precheck.json'
R_GRAM_PATH = OUTPUT_DIR / 'R_gram.npy'
R_COS_PATH = OUTPUT_DIR / 'R_cos.npy'

if CONFIG['RUN_GEOMETRY']:
    from scipy.optimize import linprog
    from src.effective_lora_geometry import effective_lora_inner_product, load_effective_lora_geometry, validate_compatible_geometries
    adapter_paths = {a: OUTPUT_DIR / 'rs_runs' / f'ppo_{a}' / 'adapter' for a in CONFIG['ATTRIBUTES']}
    missing = [str(p) for p in adapter_paths.values() if not p.is_dir()]
    assert not missing, 'Missing PPO adapters: ' + ', '.join(missing)
    geometries = {a: load_effective_lora_geometry(p) for a, p in adapter_paths.items()}
    validate_compatible_geometries([geometries[a] for a in CONFIG['ATTRIBUTES']], CONFIG['ATTRIBUTES'])
    n = len(CONFIG['ATTRIBUTES'])
    gram = np.zeros((n, n), dtype=np.float64)
    for i, left in enumerate(CONFIG['ATTRIBUTES']):
        for j, right in enumerate(CONFIG['ATTRIBUTES']):
            gram[i, j] = effective_lora_inner_product(geometries[left], geometries[right])
    gram = 0.5 * (gram + gram.T)
    norms = np.sqrt(np.maximum(np.diag(gram), 0.0))
    assert np.all(norms > 0)
    cos = gram / np.outer(norms, norms)
    np.fill_diagonal(cos, 1.0)
    offdiag = cos[~np.eye(n, dtype=bool)]
    neg = offdiag[offdiag < 0]

    c = np.zeros(2 * n + 1); c[-1] = -1.0
    A_ub, b_ub = [], []
    for i in range(n):
        row = np.zeros(2 * n + 1); row[:n] = -cos[i]; row[n:2*n] = cos[i]; row[-1] = 1.0
        A_ub.append(row); b_ub.append(0.0)
    row = np.zeros(2 * n + 1); row[:2*n] = 1.0
    A_ub.append(row); b_ub.append(1.0)
    lp = linprog(c, A_ub=np.asarray(A_ub), b_ub=np.asarray(b_ub), A_eq=np.asarray([np.r_[np.ones(n), -np.ones(n), 0.0]]), b_eq=np.asarray([0.0]), bounds=[(0.0, None)] * (2*n) + [(None, None)], method='highs')
    assert lp.success, lp.message

    write_numpy(R_GRAM_PATH, gram); write_numpy(R_COS_PATH, cos)
    write_json(DELTA_NORMS_PATH, {'attributes': CONFIG['ATTRIBUTES'], 'delta_norm': dict(zip(CONFIG['ATTRIBUTES'], norms.tolist())), 'mean_norm': float(norms.mean()), 'max_percent_deviation_from_mean': float(np.max(np.abs(norms / norms.mean() - 1.0)) * 100.0)})
    write_json(GEOMETRY_PRECHECK_PATH, {'attributes': CONFIG['ATTRIBUTES'], 'R_minus_nonzero': bool(len(neg) > 0), 'negative_offdiag_count': int(len(neg)), 'negative_offdiag_min': float(neg.min()) if len(neg) else None, 'cosine_offdiag_min': float(offdiag.min()), 'cosine_offdiag_max': float(offdiag.max()), 'floor_lp_l1_bound': 1.0, 'floor_lp_max_min_Rv': float(-lp.fun), 'floor_collapse_risk': not bool(len(neg) > 0), 'circular_do_not_report_as_proxy_validation': True})
    display(pd.DataFrame(cos, index=CONFIG['ATTRIBUTES'], columns=CONFIG['ATTRIBUTES']))
else:
    write_json(DELTA_NORMS_PATH, {'pending': True, 'reason': 'RUN_GEOMETRY is False'})
    write_json(GEOMETRY_PRECHECK_PATH, {'pending': True, 'reason': 'RUN_GEOMETRY is False'})
    print('RUN_GEOMETRY is False; geometry skipped.')

## Phase 4 und 5 - Wall-A-Test und Merge-Test

Diese kompakte Zelle baut die Suchmenge B und wertet vorhandene `reward_matrix.npy` aus. Die teure Reward-Collection und echte lambda*-Merge-Auswertung bleiben hinter expliziten Flags; keine Zahlen werden erfunden.

In [ ]:
from src.proxy_validation import build_search_set

SEARCH_SET_PATH = OUTPUT_DIR / 'search_set_B.npy'
REWARD_MATRIX_PATH = OUTPUT_DIR / 'reward_matrix.npy'
LINEARITY_R2_PATH = OUTPUT_DIR / 'linearity_r2.json'
MERGE_RESULTS_PATH = OUTPUT_DIR / 'merge_results.json'
LMC_CHECK_PATH = OUTPUT_DIR / 'lmc_check.json'

PREFERENCES = {
    'uniform': [0.2, 0.2, 0.2, 0.2, 0.2],
    'dominant_helpfulness': [0.5, 0.125, 0.125, 0.125, 0.125],
    'dominant_correctness': [0.125, 0.5, 0.125, 0.125, 0.125],
    'dominant_coherence': [0.125, 0.125, 0.5, 0.125, 0.125],
    'dominant_complexity': [0.125, 0.125, 0.125, 0.5, 0.125],
    'dominant_verbosity': [0.125, 0.125, 0.125, 0.125, 0.5],
    'only_helpfulness': [1.0, 0.0, 0.0, 0.0, 0.0],
    'only_correctness': [0.0, 1.0, 0.0, 0.0, 0.0],
    'only_coherence': [0.0, 0.0, 1.0, 0.0, 0.0],
    'only_complexity': [0.0, 0.0, 0.0, 1.0, 0.0],
    'only_verbosity': [0.0, 0.0, 0.0, 0.0, 1.0],
}
B = build_search_set(5, n_dirichlet=CONFIG['SEARCH_SET_DIRICHLET'], preferences=list(PREFERENCES.values()), seed=CONFIG['SEARCH_SET_SEED'])
write_numpy(SEARCH_SET_PATH, B)
print('Search set shape:', B.shape)

if REWARD_MATRIX_PATH.exists():
    Reward = np.load(REWARD_MATRIX_PATH)
    assert Reward.shape == (B.shape[0], 5)
    assert np.all(np.isfinite(Reward))
    vertex_rewards = Reward[:5]
    rows = []
    for k, axis in enumerate(CONFIG['ATTRIBUTES']):
        y = Reward[:, k]
        y_hat = B @ vertex_rewards[:, k]
        mask = np.arange(len(B)) >= 5
        ss_res = float(np.sum((y[mask] - y_hat[mask]) ** 2))
        ss_tot = float(np.sum((y[mask] - y[mask].mean()) ** 2))
        rows.append({'axis': axis, 'r2_vertex_linear_fit': float(1.0 - ss_res / ss_tot) if ss_tot > 0 else None})
    write_json(LINEARITY_R2_PATH, {'circular_do_not_report_as_proxy_validation': True, 'rows': rows, 'interpretation': 'Low or negative quality R2 means Wall A remains.'})
    display(pd.DataFrame(rows))
else:
    write_json(LINEARITY_R2_PATH, {'pending': True, 'reason': 'No reward_matrix.npy yet. Run the expensive merge/reward collection before claiming Wall A results.'})

if CONFIG['RUN_FINAL_MERGE']:
    raise RuntimeError('RUN_FINAL_MERGE requires the full reward_samples_of_lambda merge/scoring setup. Do not approximate by nearest B.')
else:
    write_json(MERGE_RESULTS_PATH, {'pending': True, 'reason': 'RUN_FINAL_MERGE is False; no fake merge numbers written.'})

if CONFIG['RUN_LMC']:
    raise RuntimeError('RUN_LMC requires reward_samples_of_lambda from the full merge/scoring setup.')
else:
    write_json(LMC_CHECK_PATH, {'pending': True, 'reason': 'RUN_LMC is False; no fake LMC numbers written.'})

## Verdict, Summary und Zip

Alle vorhandenen Artefakte werden zusammengefuehrt. Das Verdict bleibt STOP, solange Pflichtartefakte fehlen oder Kriterien nicht belegt sind.

In [ ]:
VERDICT_PATH = OUTPUT_DIR / 'verdict.json'
SUMMARY_PATH = OUTPUT_DIR / 'summary.md'
ZIP_PATH = OUTPUT_DIR / CONFIG['OUTPUT_ZIP']
required_outputs = {
    'preregistration': PREREGISTRATION_PATH,
    'head_sanity': HEAD_SANITY_PATH,
    'delta_norms': DELTA_NORMS_PATH,
    'R_gram': R_GRAM_PATH,
    'R_cos': R_COS_PATH,
    'geometry_precheck': GEOMETRY_PRECHECK_PATH,
    'lmc_check': LMC_CHECK_PATH,
    'reward_matrix': REWARD_MATRIX_PATH,
    'search_set_B': SEARCH_SET_PATH,
    'linearity_r2': LINEARITY_R2_PATH,
    'merge_results': MERGE_RESULTS_PATH,
}
loaded = {}
for key, p in required_outputs.items():
    p = Path(p)
    if p.exists() and p.suffix == '.json':
        loaded[key] = json.loads(p.read_text(encoding='utf-8'))
missing = [name for name, p in required_outputs.items() if not Path(p).exists()]
reasons = []
if missing: reasons.append('missing outputs: ' + ', '.join(missing))
if loaded.get('head_sanity', {}).get('passed') is not True: reasons.append('head sanity not passed')
geom = loaded.get('geometry_precheck', {})
if geom.get('pending'): reasons.append('geometry pending')
elif geom.get('R_minus_nonzero') is not True: reasons.append('R-minus not detected; conflict-free geometry may persist')
lin = loaded.get('linearity_r2', {})
if lin.get('pending'): reasons.append('linearity R2 pending')
merge = loaded.get('merge_results', {})
if merge.get('pending'): reasons.append('final merge test pending')

verdict = {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'circularity_acknowledged': True, 'retired_rqs': ['RQ2 proxy validity'], 'valid_claims': ['upper bound', 'R-minus', 'wall A (R2)', 'LMC'], 'invalid_claims': ['proxy validity', 'generalization beyond this reward model'], 'circular_do_not_report_as_proxy_validation': True, 'decision': 'GO' if not reasons else 'STOP', 'reasons': reasons, 'config': CONFIG}
write_json(VERDICT_PATH, verdict)

summary = '# Was dieses Notebook zeigen kann und was nicht\n\n'
summary += '**Gueltig:** obere Schranke, R-minus, Wall A/R2, LMC.  \n'
summary += '**Ungueltig:** RQ2 proxy validity und Generalisierung ueber ArmoRM hinaus.\n\n'
summary += 'Dieses Notebook trainiert PPO bewusst gegen ArmoRM und evaluiert gegen ArmoRM. Das ist zirkulaer und in `preregistration.json` explizit anerkannt. Spearman- oder Gate-Zahlen aus diesem Lauf duerfen nicht als Proxy-Validierung berichtet werden.\n\n'
summary += f"## Verdict\n\nDecision: **{verdict['decision']}**\n\nReasons:\n"
summary += '\n'.join('- ' + r for r in reasons) if reasons else '- all preregistered criteria satisfied'
summary += '\n\n## Outputs\n\n' + '\n'.join('- ' + name + ': ' + str(p) for name, p in required_outputs.items()) + '\n'
SUMMARY_PATH.write_text(summary, encoding='utf-8')

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for p in [*required_outputs.values(), VERDICT_PATH, SUMMARY_PATH]:
        p = Path(p)
        if p.exists():
            archive.write(p, arcname=p.name)
            print('Added:', p.name)
print('Output zip:', ZIP_PATH)
print('Verdict:', verdict['decision'], verdict['reasons'])
try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except Exception as error:
    print('files.download is available only in Colab. Download manually from:', ZIP_PATH)
    print('Download error:', repr(error))